In [ ]:
from supabase import create_client
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

# Your Supabase creds (Dashboard > Settings > API)
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Load cleaned (200k rows)
df = pd.read_csv('../data/transactions_clean.csv')

# Batch insert (5k rows/chunk → ~40 calls, <5min)
success = 0
for i in range(0, len(df), 10000):
    chunk = df.iloc[i:i+10000].to_dict('records')
    try:
        response = supabase.table('transactions_staging').insert(chunk).execute()
        success += len(chunk)
        print(f"Inserted {len(chunk)} rows (total: {success})")
    except Exception as e:
        print(f"Error at {i}: {e}")

print("Upload complete!")
